# Assignment 4: Bank Customer Churn Prediction using a Neural Network
Dataset: `Churn_Modelling.csv` (Kaggle: barelydedicated/bank-customer-churn-modeling). Keep it in the same folder as this notebook.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(42)
np.random.seed(42)

## Step 1: Read the dataset

In [ ]:
df = pd.read_csv("Churn_Modelling.csv")
print(df.shape)
print(df.isnull().sum().sum(), "missing values")
df.head()

In [ ]:
sns.countplot(x=df["Exited"])
plt.title("Churn distribution (0 = stays, 1 = leaves)")
plt.show()
print(df["Exited"].value_counts(normalize=True).round(3))

## Step 2: Separate features and target, split into train and test sets

In [ ]:
# RowNumber, CustomerId and Surname do not help prediction
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Encode categorical columns (Geography, Gender)
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=True, dtype=int)

X = df.drop("Exited", axis=1)
y = df["Exited"]
print(list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## Step 3: Normalize the train and test data

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit on train only
X_test = scaler.transform(X_test)

## Step 4a: Initialize and build the base model

In [ ]:
base = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(6, activation="relu"),
    layers.Dense(6, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
base.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
base.summary()
base_hist = base.fit(X_train, y_train, validation_split=0.2, epochs=50, batch_size=32, verbose=0)

base_pred = (base.predict(X_test, verbose=0) > 0.5).astype(int).ravel()
print("Base model accuracy:", round(accuracy_score(y_test, base_pred)*100, 2), "%")
print(confusion_matrix(y_test, base_pred))

## Step 4b: Points of improvement and implementation
The base model can be improved by:
1. A **bigger network** (more neurons / one more hidden layer) so it can learn more complex patterns.
2. **Dropout** to reduce overfitting.
3. **Early stopping** so training stops when validation loss stops improving.
4. **Class weights**, because only about 20% of customers leave and the model otherwise ignores them (low recall for churners).

In [ ]:
improved = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid"),
])
improved.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                 loss="binary_crossentropy", metrics=["accuracy"])

cw = compute_class_weight("balanced", classes=np.array([0, 1]), y=y_train)
class_weight = {0: cw[0], 1: cw[1]}
print("Class weights:", class_weight)

early_stop = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
hist = improved.fit(X_train, y_train, validation_split=0.2, epochs=150, batch_size=32,
                    class_weight=class_weight, callbacks=[early_stop], verbose=0)
print("Stopped after", len(hist.history["loss"]), "epochs")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(hist.history["loss"], label="train"); ax[0].plot(hist.history["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(hist.history["accuracy"], label="train"); ax[1].plot(hist.history["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.show()

## Step 5: Accuracy score and confusion matrix

In [ ]:
y_prob = improved.predict(X_test, verbose=0).ravel()
y_pred = (y_prob > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
print("Improved model accuracy:", round(acc*100, 2), "%")
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Stays", "Leaves"], yticklabels=["Stays", "Leaves"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix (Improved Model)")
plt.show()
print(classification_report(y_test, y_pred, target_names=["Stays", "Leaves"]))

In [ ]:
print("Base model     accuracy:", round(accuracy_score(y_test, base_pred)*100, 2), "%")
print("Improved model accuracy:", round(acc*100, 2), "%")
print("\nBase model confusion matrix:\n", confusion_matrix(y_test, base_pred))
print("Improved model confusion matrix:\n", cm)